In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import vstack
from collections import Counter

#Load news dataset with the columns
news_df = pd.read_csv(
    "../MINDsmall_train/news.tsv",
    sep="\t",
    header=None,
    names=[
        "news_id", "category", "subcategory",
        "title", "abstract", "url",
        "title_entities", "abstract_entities"
    ]
)

#Preprocess the data to fill missing values 
news_df["title"]    = news_df["title"].fillna("")
news_df["abstract"] = news_df["abstract"].fillna("")
news_df["url"]      = news_df["url"].fillna("")

#Select required columns
news_df = news_df[["news_id", "category", "subcategory", "title", "abstract", "url"]]

#Combine title and abstract column
news_df["content"] = news_df["title"] + " " + news_df["abstract"]

#Load behaviors with the columns
behaviors_df = pd.read_csv(
    "../MINDsmall_train/behaviors.tsv",
    sep="\t",
    header=None,
    names=["impression_id", "user_id", "time", "history", "impressions"]
)

#Preprocess the data to fill missing values and split history into a list

behaviors_df["history"] = behaviors_df["history"].fillna("")
behaviors_df["history"] = behaviors_df["history"].apply(lambda x: x.split())

#Create a TD-IDF matrix for the news content
tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = tfidf.fit_transform(news_df["content"])

#Map NEWS_ID
news_id_to_index = {nid: idx for idx, nid in enumerate(news_df["news_id"])}

#Recency score
news_df["recency_score"] = news_df.index / len(news_df)

print(f"News loaded       : {len(news_df):,} articles")
print(f"Behaviors loaded  : {len(behaviors_df):,} records")
print(f"TF-IDF matrix     : {tfidf_matrix.shape}")